# C3-Z26 C3-ROI V2 — artifact pilots B/C
Fold 1, seed 42, direct regression. B là artifact augmentation; C thêm consistency loss.


In [ ]:
PILOT = 'B'  # đổi thành 'C' sau khi B kết thúc
assert PILOT in {'B', 'C'}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile, shutil, subprocess, sys
DATA_DIR = Path('/content/drive/MyDrive/RSNA_DATA')
MAIN_CODE_ZIP = DATA_DIR / 'C3_Z26_C3_ROI_T4_CODE_V2.zip'
DATA_ZIP = DATA_DIR / 'C3_Z26_COMBO_V2_FINAL.zip'
PILOT_ZIP = DATA_DIR / 'C3_Z26_C3_ROI_V2_PILOT_BC_CODE_V2.zip'
for path in (MAIN_CODE_ZIP, DATA_ZIP, PILOT_ZIP):
    assert path.is_file(), f'Thiếu {path}'
if not Path('/content/C3_Z26_C3_ROI_V2/manifests/fold_1_train.csv').is_file():
    print('Extract main C3-ROI V2 code...')
    with zipfile.ZipFile(MAIN_CODE_ZIP) as zf: zf.extractall('/content')
if not Path('/content/C3_Z26_COMBO_V2').is_dir():
    print('Extract C3-Z26 image data...')
    with zipfile.ZipFile(DATA_ZIP) as zf: zf.extractall('/content')
shutil.rmtree('/content/p1_baseline', ignore_errors=True)
with zipfile.ZipFile(PILOT_ZIP) as zf: zf.extractall('/content')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/requirements_colab.txt'], check=True)
sys.path.insert(0, '/content')
assert Path('/content/C3_Z26_C3_ROI_V2/manifests/fold_1_train.csv').is_file()
assert Path('/content/C3_Z26_COMBO_V2').is_dir()
print('SETUP PASS')


In [ ]:
# Runner tự preflight, chọn checkpoint tương thích và resume từ Drive nếu Colab ngắt.
runner = Path('/content/C3_Z26_C3_ROI_V2_PILOTS/pilot_bc_runner.py')
subprocess.run([sys.executable, '-u', str(runner), '--pilot', PILOT], check=True)


In [ ]:
# Đánh giá paired clean/artifact trên Fold 1; không đọc tập test 200 ảnh.
evaluator = Path('/content/C3_Z26_C3_ROI_V2_PILOTS/evaluate_pilot_bc.py')
subprocess.run([sys.executable, '-u', str(evaluator), '--pilot', PILOT], check=True)


In [ ]:
run_id = f'C3_Z26_C3_ROI_V2_PILOT_{PILOT}_V2_FOLD_1_SEED_42'
run_dir = DATA_DIR / 'C3_Z26_C3_ROI_V2_PILOTS' / 'runs' / run_id
print((run_dir / 'train.log').read_text(encoding='utf-8')[-4000:])
print((run_dir / 'artifact_robustness_report.json').read_text(encoding='utf-8'))
print('Best checkpoint:', run_dir / 'best_mae.ckpt')
